In [1]:
# FPGA ML Pollution Prediction Project
# Stage: Model Training
# Target FPGA: Basys-3 (Artix-7)

# Task: Predict PM25(t+1)
# Window size: 6 hours
# Input dimension: 54
# Output dimension: 1

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(54,)),

    layers.Dense(16, activation='relu', name="dense_1"),
    layers.Dense(8, activation='relu', name="dense_2"),
    layers.Dense(1, activation='linear', name="output")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 16)             │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,025 (4.00 KB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
X = np.load("data/X_windows.npy")
y = np.load("data/y_targets.npy")

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20323, 54)
y shape: (20323,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Train samples: 16258
Test samples: 4065


In [3]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [7]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    shuffle=False
)

Epoch 1/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 461.7212 - mae: 11.8360 - val_loss: 699.6833 - val_mae: 20.0662
Epoch 2/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 317.0977 - mae: 10.0066 - val_loss: 439.0777 - val_mae: 16.7312
Epoch 3/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 300.0360 - mae: 9.7411 - val_loss: 539.9385 - val_mae: 17.4710
Epoch 4/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 289.6597 - mae: 9.5785 - val_loss: 465.6343 - val_mae: 16.0425
Epoch 5/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 296.2853 - mae: 9.5335 - val_loss: 467.9929 - val_mae: 16.6862
Epoch 6/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 273.8489 - mae: 9.2600 - val_loss: 353.4049 - val_mae: 13.5662
Epoch 7/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 281.5475 - mae: 9.2608 - val_loss: 367.1002 - val_mae: 14.4170
Epoch 8/20
509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 882us/step - loss: 266.2467 - mae: 9.1145 - val_loss: 322.1461 - val_mae: 13.4779
Epoch 9/20
5

In [8]:
test_loss, test_mae = model.evaluate(X_test, y_test)

print("Final Test MSE:", test_loss)
print("Final Test MAE:", test_mae)

128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 335.2346 - mae: 13.3173
Final Test MSE: 335.2346496582031
Final Test MAE: 13.317275047302246


In [9]:
model.save("model_fpga_ready.h5")